# 02 · Silver Build

Reads all rows from `bronze/prices/`, applies cleaning and enrichment, and incrementally
merges into `asset_mgmt.silver.daily_prices` (Delta table).

**Idempotent** — safe to re-run. MERGE INTO updates a row only when the incoming
`_ingest_timestamp` is newer than what is already in Silver.

| Step | What happens |
|---|---|
| Cast | Enforce correct Spark types on every column |
| Validate | Drop rows where `Close` is null |
| Flag | Mark rows where `Volume < 0` with `volume_flag = true` |
| Enrich | Join sector lookup (inline) |
| Compute | `daily_return` = (Close − prev Close) / prev Close per ticker |
| Merge | MERGE INTO on (Ticker, Date), keep latest `_ingest_timestamp` |


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, LongType, DateType, StringType, BooleanType

BRONZE_PATH  = "/Volumes/asset_mgmt/bronze/prices/"
SILVER_TABLE = "asset_mgmt.silver.daily_prices"

print(f"Bronze path  : {BRONZE_PATH}")
print(f"Silver table : {SILVER_TABLE}")

In [ ]:
# ── Create Silver table if it doesn't exist ───────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    Ticker           STRING  COMMENT 'Stock ticker symbol',
    Date             DATE    COMMENT 'Trading date',
    Open             DOUBLE  COMMENT 'Opening price (USD)',
    High             DOUBLE  COMMENT 'Intraday high price (USD)',
    Low              DOUBLE  COMMENT 'Intraday low price (USD)',
    Close            DOUBLE  COMMENT 'Closing price, adjusted (USD)',
    Volume           LONG    COMMENT 'Number of shares traded',
    Sector           STRING  COMMENT 'GICS sector classification',
    daily_return     DOUBLE  COMMENT 'Day-over-day return: (Close - prev_Close) / prev_Close',
    volume_flag      BOOLEAN COMMENT 'True if Volume < 0 (data quality issue)',
    _ingest_timestamp TIMESTAMP COMMENT 'When this row was ingested into Bronze',
    _source_type     STRING  COMMENT 'Ingestion source: seed | batch_2 | live',
    _source_file     STRING  COMMENT 'Source Parquet file path'
)
USING DELTA
COMMENT 'Cleaned and enriched daily OHLCV prices — one row per (Ticker, Date)'
""")

print(f"Table ready: {SILVER_TABLE}")

In [ ]:
# ── Read all Bronze data ──────────────────────────────────────────────────
bronze = spark.read.parquet(BRONZE_PATH)

print(f"Bronze rows : {bronze.count():,}")
bronze.printSchema()

In [ ]:
# ── Type casting + validation ─────────────────────────────────────────────
before = bronze.count()

casted = (
    bronze
    .withColumn("Ticker",  F.col("Ticker").cast(StringType()))
    .withColumn("Date",    F.col("Date").cast(DateType()))
    .withColumn("Open",    F.col("Open").cast(DoubleType()))
    .withColumn("High",    F.col("High").cast(DoubleType()))
    .withColumn("Low",     F.col("Low").cast(DoubleType()))
    .withColumn("Close",   F.col("Close").cast(DoubleType()))
    .withColumn("Volume",  F.col("Volume").cast(LongType()))
)

# Drop rows where Close is null (unrecoverable)
cleaned = casted.filter(F.col("Close").isNotNull())

# Flag negative volumes without dropping them
cleaned = cleaned.withColumn("volume_flag", F.col("Volume") < 0)

after = cleaned.count()
print(f"Rows before  : {before:,}")
print(f"Rows dropped : {before - after:,}  (null Close)")
print(f"Rows after   : {after:,}")
print(f"Volume flags : {cleaned.filter(F.col('volume_flag')).count():,}")

In [ ]:
# ── Sector lookup (inline — no external CSV needed) ───────────────────────
sector_rows = [
    ("AAPL", "Tech"),  ("MSFT", "Tech"),  ("NVDA", "Tech"),  ("GOOGL", "Tech"),  ("META", "Tech"),
    ("JPM",  "Finance"),("BAC",  "Finance"),("GS",   "Finance"),("MS",   "Finance"),("WFC",  "Finance"),
    ("JNJ",  "Healthcare"),("UNH","Healthcare"),("PFE","Healthcare"),("ABBV","Healthcare"),("MRK","Healthcare"),
    ("XOM",  "Energy"), ("CVX",  "Energy"), ("COP",  "Energy"), ("SLB",  "Energy"), ("EOG",  "Energy"),
    ("AMZN", "Consumer"),("HD",  "Consumer"),("MCD",  "Consumer"),("NKE",  "Consumer"),("COST", "Consumer"),
    ("SPY",  "Benchmark"),
]
sector_df = spark.createDataFrame(sector_rows, ["Ticker", "Sector"])

with_sector = cleaned.join(sector_df, on="Ticker", how="left")

# Sanity check: any tickers without a sector?
unmapped = with_sector.filter(F.col("Sector").isNull()).select("Ticker").distinct()
if unmapped.count() > 0:
    print("[WARN] Tickers missing sector mapping:")
    unmapped.show()
else:
    print("All tickers mapped to a sector.")

In [ ]:
# ── Daily return per ticker ───────────────────────────────────────────────
# Window over full history so the lag always has a previous row to look back at.
# The first trading day for each ticker will have daily_return = null (expected).
ticker_window = Window.partitionBy("Ticker").orderBy("Date")

with_return = (
    with_sector
    .withColumn("prev_close", F.lag("Close", 1).over(ticker_window))
    .withColumn(
        "daily_return",
        F.when(
            F.col("prev_close").isNotNull() & (F.col("prev_close") != 0),
            (F.col("Close") - F.col("prev_close")) / F.col("prev_close")
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    .drop("prev_close")
)

print(f"Rows with daily_return : {with_return.filter(F.col('daily_return').isNotNull()).count():,}")
print(f"Rows without (first day per ticker) : {with_return.filter(F.col('daily_return').isNull()).count():,}")

In [ ]:
# ── Select final columns (must match Silver table schema exactly) ──────────
staged = with_return.select(
    "Ticker", "Date", "Open", "High", "Low", "Close", "Volume",
    "Sector", "daily_return", "volume_flag",
    "_ingest_timestamp", "_source_type", "_source_file"
)

staged.createOrReplaceTempView("staged")
print(f"Staged rows : {staged.count():,}")
display(staged.limit(5))

In [ ]:
# ── MERGE INTO silver.daily_prices ────────────────────────────────────────
spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING staged AS source
  ON target.Ticker = source.Ticker
 AND target.Date   = source.Date
WHEN MATCHED AND source._ingest_timestamp > target._ingest_timestamp
  THEN UPDATE SET *
WHEN NOT MATCHED
  THEN INSERT *
""")

print("MERGE complete.")

In [ ]:
# ── Verify ────────────────────────────────────────────────────────────────
silver = spark.table(SILVER_TABLE)

print(f"Silver rows total : {silver.count():,}")

display(
    silver
    .groupBy("Sector")
    .agg(
        F.countDistinct("Ticker").alias("tickers"),
        F.count("*").alias("rows"),
        F.min("Date").alias("date_min"),
        F.max("Date").alias("date_max"),
        F.round(F.avg("daily_return") * 100, 4).alias("avg_daily_return_pct"),
    )
    .orderBy("Sector")
)